# 46. CD-OPE-S 통계검정과 Seed 안정성

sample-level bootstrap보다 seed-level paired delta를 우선합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch4_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/4장/ch4_utils.py")) + list(Path.cwd().glob("**/ch4_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "4장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch4_utils import *

paths = find_ch4_paths()
set_korean_font()
set_seed(41)
paths

Chapter4Paths(chapter4_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장'), chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/manifests'), design_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/chapter4_cd_ope_s_architecture_design.md'), validity_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/chapter4_cd_ope_s_design_validity_review.md'))

## 46-1. seed metrics와 baseline reference 수집

In [2]:
seed_metrics = collect_ch4_cd_ope_metrics()
if seed_metrics.empty:
    raise FileNotFoundError("42번에서 CD-OPE-S 학습 run을 먼저 생성하세요.")
display(seed_metrics)

stat_paths = build_ch4_seed_statistics()
stat_paths

,variant,model_seed,run_dir,mean_dice,seen_color_dice,heldout_color_dice,red_dice,purple_dice,worst_combo_dice,target_fnr,alpha_global,alpha_local
0,g_cd,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.555547,0.558469,0.551165,0.530599,0.571731,0.160530,0.488617,0.061157,0.000000
1,g_cd,1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.593736,0.621262,0.552447,0.495976,0.608918,0.205672,0.429164,0.040685,0.000000
2,g_cd,2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.511495,0.526553,0.488908,0.478978,0.498838,0.101797,0.540872,0.056230,0.000000
3,gl_cd,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.525958,0.560491,0.474159,0.411909,0.536410,0.189626,0.529936,0.052356,0.022873
4,gl_cd,1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.485473,0.535433,0.410532,0.382798,0.438267,0.142344,0.562911,0.045000,-0.060026
5,gl_cd,2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.501366,0.514625,0.481477,0.488751,0.474204,0.143862,0.547570,0.065900,0.026714
6,gl_cd_consistency,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.603256,0.626133,0.568940,0.503065,0.634816,0.240694,0.436429,0.051055,0.025223
7,gl_cd_consistency,1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.566173,0.595692,0.521896,0.486591,0.557201,0.181444,0.478784,0.041712,-0.065609
8,gl_cd_consistency,2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.608428,0.619006,0.592562,0.581388,0.603735,0.232915,0.417324,0.069178,0.021453
9,gl_cd_consistency_style,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.557728,0.578172,0.527062,0.481942,0.572182,0.164774,0.498809,0.066665,0.021097


{'baseline': WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/cd_ope_s/baseline_reference_seed_metrics.csv'),
 'deltas': WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/cd_ope_s/paired_seed_deltas.csv'),
 'summary': WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/cd_ope_s/paired_seed_delta_summary.csv')}

## 46-2. paired delta 해석

In [3]:
deltas = pd.read_csv(stat_paths["deltas"])
display(deltas)
if stat_paths["summary"].exists():
    summary = pd.read_csv(stat_paths["summary"])
    display(summary)
    display(summary.pivot_table(index="variant", columns="metric", values="mean"))

,variant,model_seed,metric,delta
0,g_cd,0,red_dice,0.530599
1,g_cd,0,heldout_color_dice,0.313040
2,g_cd,0,seen_color_dice,-0.096311
3,g_cd,0,worst_combo_dice,0.160530
4,g_cd,0,target_fnr,-0.043761
5,g_cd,1,red_dice,0.393877
6,g_cd,1,heldout_color_dice,0.177612
7,g_cd,1,seen_color_dice,-0.043303
8,g_cd,1,worst_combo_dice,0.205672
9,g_cd,1,target_fnr,-0.073028


,variant,metric,mean,std,count,ci95_half_width
0,g_cd,heldout_color_dice,0.212510,0.088407,3,0.100042
1,g_cd,red_dice,0.444673,0.074825,3,0.084673
2,g_cd,seen_color_dice,-0.104566,0.065780,3,0.074437
3,g_cd,target_fnr,-0.015589,0.075572,3,0.085518
4,g_cd,worst_combo_dice,0.156000,0.052085,3,0.058940
5,gl_cd,heldout_color_dice,0.137060,0.100190,3,0.113376
6,gl_cd,red_dice,0.370641,0.077981,3,0.088243
7,gl_cd,seen_color_dice,-0.136477,0.046300,3,0.052393
8,gl_cd,target_fnr,0.044999,0.041857,3,0.047366
9,gl_cd,worst_combo_dice,0.158611,0.026871,3,0.030407


metric,heldout_color_dice,red_dice,seen_color_dice,target_fnr,worst_combo_dice
variant,,,,,
g_cd,0.212510,0.444673,-0.104566,-0.015589,0.156000
gl_cd,0.137060,0.370641,-0.136477,0.044999,0.158611
gl_cd_consistency,0.242803,0.466504,-0.059717,-0.057627,0.218351
gl_cd_consistency_style,0.217500,0.445737,-0.087569,-0.016727,0.166242


## 46-3. 성공 조건 체크

In [4]:
if stat_paths["summary"].exists():
    summary = pd.read_csv(stat_paths["summary"])
    target = summary[(summary["variant"] == "gl_cd_consistency_style") & (summary["metric"] == "red_dice")]
    if not target.empty:
        mean_delta = float(target["mean"].iloc[0])
        print("A5 red_dice paired mean delta:", mean_delta)
        print("판정:", "baseline 대비 red 개선" if mean_delta > 0 else "red 개선 불충분")

A5 red_dice paired mean delta: 0.4457368512352793
판정: baseline 대비 red 개선
